# Leson 11 - Agent-to-Agent (A2A) Protocol


## How to set am


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Wetin be the A2A Protocol?

The **Agent-to-Agent (A2A) protocol** na open standard wey enable AI agents to communicate,
discover each oda, and collaborate — even when dem built dem for different frameworks or hosted
by different services.

Key concepts:

- **Discovery** – Agents publish an *Agent Card* wey describe their capabilities, making am
  easy for oda agents (or orchestrators) to find the right specialist for a task.
- **Message Passing** – Agents dey exchange structured messages through a common protocol, so a
  request from one agent fit be understood and fulfilled by another regardless of how e take
  implement inside.
- **Task Lifecycle** – A2A dey define states such as *submitted*, *working*, *completed*, and
  *failed*, wey give the orchestrator full visibility into how a delegated task dey progress.

For this lesson we go simulate A2A-style collaboration by wiring three specialized travel agents
into a workflow where each agent go contribute im expertise and pass results to the next.


## How to make travel agents wey get special skills


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## How Multiple Agents Dey Collaborate Through Workflow

We connect di three agents into one sequential workflow wey dey mirror A2A message passing:

1. **CurrencyExchangeAgent** dey receive di user request an produce currency guidance.
2. **ActivityPlannerAgent** dey receive di enriched context an add activity recommendations.
3. **TravelManagerAgent** dey synthesize both inputs into one final travel brief.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## How A2A dey work for Production

For production environment the A2A protocol dey unlock strong cross-service scenarios:

| Wetin e fit do | Tok |
|---|---|
| **Cross-framework interop** | One agent wey dem build with one framework fit give tasks to another agent wey dem build with any other A2A-compliant framework, so organisations fit really interoperate. |
| **Service boundaries** | Agents fit dey for different microservices, cloud regions, or even different organisations, but dem go still collaborate without wahala. |
| **Dynamic discovery** | An orchestrator fit query an Agent Card registry during runtime to find di best specialist for any sub-task. |
| **Streaming & push notifications** | A2A dey support Server-Sent Events (SSE) for real-time progress updates and push notifications for long-running tasks. |

The workflow wey we build above na simplified, in-process version of this pattern. In a real deployment each agent go expose an HTTP endpoint, publish an Agent Card, and communicate via the A2A JSON-RPC protocol.


## Summary

For dis lesson you don learn:

1. **Wetin di A2A protocol be** — na open standard wey make agents fit find one another, send messages,
   and manage tasks.
2. **How you go create specialized agents** — a Currency Exchange agent, an Activity Planner agent,
   and a Travel Manager orchestrator.
3. **How you go wire agents into a workflow** — using `WorkflowBuilder` to model sequential
   message passing between agents.
4. **How A2A dey work in production** — enabling cross-framework, cross-service collaboration
   with dynamic discovery and streaming updates.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dis document don translate wit AI translation service Co-op Translator (https://github.com/Azure/co-op-translator). Even though we dey try make am correct, abeg sabi say automated translations fit get mistakes or wrong tok. Di original document for im own language na di authority one. If na important information, make una use professional human translator. We no go responsible for any misunderstanding or wrong interpretation wey fit come from dis translation.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
